# Week 3 lecture walkthrough: Čech and Rips filtrations

This is the live worked example, built around three equal-radius balls followed by one sampled circle. It follows the conceptual argument of the slides through prediction, reveal and interpretation. It is not intended as a line-by-line answer key to the participant practical.

**Resource boundary.** The [reference notes](index.qmd) explain filtrations, Čech, Rips and the Nerve Theorem. The [slides](slides.qmd) stage the construction through scale. This notebook reveals the worked three-point calculation. The [participant practical](lab.ipynb) leaves scale choices and the final comparison to students.

**Lecture map.** Keep the three points and radius fixed while changing only the Čech-versus-Rips rule. Then keep one sampled circle fixed and vary scale. These two comparisons separate a construction choice from a filtration change.

This practical keeps the points fixed and changes one construction choice at a time:

$$S\subset\mathbb R^2 \longrightarrow \{B(x,r)\}_{x\in S}
\longrightarrow \check C_r(S)\ \text{or}\ \operatorname{Rips}_r(S)
\longrightarrow H_p(-;\mathbb F_2).$$

We use the **ball-radius convention**: Čech uses balls of radius $r$, while Rips contains simplices of diameter at most $2r$. With this convention the two complexes have the same edges and
$\check C_r(S)\subseteq\operatorname{Rips}_r(S)$.

**† Qualification.** Many libraries instead call the pairwise-distance threshold itself the Rips parameter. Always check whether a reported value is a radius $r$ or a diameter threshold $2r$.

**Presenter route.** Use the three-ball example as the central reveal: pairwise intersection creates a Rips face before joint intersection creates the Čech face.



In [ ]:
from itertools import combinations
import numpy as np
import matplotlib.pyplot as plt

def faces(simplex):
    return [tuple(f) for k in range(1, len(simplex)) for f in combinations(simplex, k)]

def close_under_faces(maximal):
    K = set()
    for s in maximal:
        s = tuple(sorted(s)); K.add(s); K.update(faces(s))
    return K

def boundary_matrix(K, p):
    cols = sorted(s for s in K if len(s) == p + 1)
    rows = sorted(s for s in K if len(s) == p) if p else []
    A = np.zeros((len(rows), len(cols)), dtype=np.uint8)
    lookup = {s:i for i,s in enumerate(rows)}
    if p:
        for j,s in enumerate(cols):
            for f in combinations(s, p): A[lookup[f],j] = 1
    return A

def rank_mod2(A):
    A=np.array(A,dtype=np.uint8,copy=True)%2; row=rank=0
    for col in range(A.shape[1]):
        piv=np.flatnonzero(A[row:,col])
        if not len(piv): continue
        q=row+piv[0]; A[[row,q]]=A[[q,row]]
        for i in range(A.shape[0]):
            if i!=row and A[i,col]: A[i]^=A[row]
        row+=1; rank+=1
        if row==A.shape[0]: break
    return rank

def betti(K, max_dim=1):
    out=[]
    for p in range(max_dim+1):
        n=sum(len(s)==p+1 for s in K)
        rp=rank_mod2(boundary_matrix(K,p)) if p else 0
        rn=rank_mod2(boundary_matrix(K,p+1))
        out.append(n-rp-rn)
    return tuple(out)

def enclosing_radius(points):
    """Radius of the smallest enclosing ball for one, two or three planar points."""
    P=np.asarray(points,float)
    if len(P)==1: return 0.0
    d=np.array([np.linalg.norm(P[i]-P[j]) for i,j in combinations(range(len(P)),2)])
    if len(P)==2: return d[0]/2
    a,b,c=sorted(d)
    if c*c >= a*a+b*b-1e-12: return c/2
    area=abs(np.cross(P[1]-P[0],P[2]-P[0]))/2
    return a*b*c/(4*area)

def rips(points, r, max_dim=2):
    """Rips(r): simplices of diameter at most 2r."""
    K=set()
    for k in range(1,max_dim+2):
        for s in combinations(range(len(points)),k):
            diameter=max([0.0]+[np.linalg.norm(points[i]-points[j]) for i,j in combinations(s,2)])
            if diameter <= 2*r+1e-12: K.add(s)
    return K

def cech(points, r, max_dim=2):
    """Cech(r): balls of radius r centred at the vertices have common intersection."""
    K=set()
    for k in range(1,max_dim+2):
        for s in combinations(range(len(points)),k):
            if enclosing_radius(points[list(s)]) <= r+1e-12: K.add(s)
    return K

def graph_complex(points, r):
    """Only vertices and Rips edges, with no clique filling."""
    return {s for s in rips(points,r,max_dim=1)}

def valid_filtration_value(values):
    violations=[]
    for s,v in values.items():
        for f in faces(s):
            if f in values and values[f] > v: violations.append((f,s))
    return violations

def draw(K, points, ax, title):
    for tri in sorted(s for s in K if len(s)==3):
        ax.fill(*zip(*points[list(tri)]),color='tab:blue',alpha=.22)
    for e in sorted(s for s in K if len(s)==2):
        ax.plot(*zip(*points[list(e)]),color='black',lw=1.5)
    ax.scatter(points[:,0],points[:,1],s=55,zorder=3)
    for i,p in enumerate(points): ax.text(p[0],p[1]+.08,str(i),ha='center')
    ax.set_title(title); ax.set_aspect('equal'); ax.axis('off')

print('Week 3 helpers ready. Homology is computed over F_2.')
print('Complexes are built through dimension 2, so reported Betti numbers are beta_0 and beta_1.')

## 1. Observe: lecture demonstration

**Presenter cue.** Show the object before the calculation. Ask the room to separate what is given from what will be constructed.

The three points below form an equilateral triangle of side length $1.9$. We examine radius $r=1$. Pairwise balls intersect because every pair of centres is at distance below $2r$. Before constructing a complex, ask the genuinely joint question: do all three balls share one point?

In [ ]:
triangle=np.array([[0.,0.],[1.9,0.],[0.95,1.9*np.sqrt(3)/2]])
r=1.0
theta=np.linspace(0,2*np.pi,300)
fig,ax=plt.subplots(figsize=(5,5))
for i,p in enumerate(triangle):
    ax.plot(p[0]+r*np.cos(theta),p[1]+r*np.sin(theta),alpha=.75)
    ax.scatter(*p); ax.text(p[0],p[1]+.08,str(i),ha='center')
ax.set_aspect('equal'); ax.set_title('Three radius-1 balls'); ax.axis('off'); plt.show()
print('pairwise distances:',np.round([np.linalg.norm(triangle[i]-triangle[j]) for i,j in combinations(range(3),2)],3))
print('smallest common enclosing radius:',round(enclosing_radius(triangle),3))

## 2. Predict: lecture demonstration

**Presenter cue.** Pause here and collect at least two predictions before revealing any output.

Before running the construction:

1. Which vertices and edges should appear in both complexes at $r=1$?
2. Should the 2-simplex $(0,1,2)$ appear in Čech, Rips, both, or neither?
3. Predict $\beta_1$ for the graph alone, the Čech complex, and the full Rips complex.
4. If $r$ increases, can an existing simplex disappear from either filtration? Explain using the definition.

The code constructs simplices only through dimension 2. This is sufficient for $H_0$ and $H_1$, which are the quantities reported here; it is not sufficient for a valid $H_2$ calculation because omitted 3-simplices could fill 2-cycles.

**Worked prediction.** All vertices and edges appear in both complexes. The triple has diameter $1.9<2r$, so Rips fills it. Its smallest enclosing radius is $1.9/\sqrt{3}\approx1.097>1$, so the three balls have no common intersection and Čech does not fill it. Therefore the graph and Čech complex have $\beta_1=1$, while the Rips complex has $\beta_1=0$. Increasing $r$ can only add simplices.

## 3. Implement: lecture demonstration

**Reveal.** Run one cell at a time. Name the domain, codomain, complex, module or summary before interpreting its values.

Čech records common intersections of balls. Rips checks only pairwise distances and fills every clique. The following calculation exposes the difference without calling a TDA library.

In [ ]:
C=cech(triangle,r); R=rips(triangle,r); G=graph_complex(triangle,r)
for name,K in [('graph',G),('Cech',C),('Rips',R)]:
 print(name,sorted(K,key=lambda s:(len(s),s)),'Betti',betti(K))

The 1-skeletons agree. The Rips rule adds the 2-simplex because its decision is completely determined by the three edges. Čech asks for a common intersection of all three balls, which is a stronger condition.

### Check the filtration property

In [ ]:
bad={(0,):0.,(1,):0.,(2,):0.,(0,1):1.,(0,2):1.,(1,2):1.4,(0,1,2):1.2}
print('violations:',valid_filtration_value(bad))
fixed=dict(bad); fixed[(0,1,2)]=max(fixed[f] for f in faces((0,1,2)))
print('repaired triangle value:',fixed[(0,1,2)],'violations:',valid_filtration_value(fixed))

The triangle cannot enter at $1.2$ while its edge $(1,2)$ waits until $1.4$. Assigning the maximum face value is the smallest repair. Equal filtration values are allowed: a simplex and some faces can enter together.

### Follow one point cloud through scale

In [ ]:
angles=np.linspace(0,2*np.pi,8,endpoint=False); circle=np.c_[np.cos(angles),np.sin(angles)]
radii=[0.20,0.39,0.72,1.01]
fig,axes=plt.subplots(1,4,figsize=(14,3.2))
for ax,q in zip(axes,radii):
 K=rips(circle,q); draw(K,circle,ax,f'r={q}; beta={betti(K)[:2]}'); print(q,betti(K))
plt.show()

At the smallest radius the vertices are separate. Once adjacent points connect, one graph cycle appears. Later, new diagonals create triangles that fill the central class. A Betti number at one chosen radius hides this evolution, which is why the nested family matters.

## 4. Compare: lecture demonstration

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(10,3.2))
for ax,(name,K) in zip(axes,[('graph',G),('Cech',C),('Rips',R)]): draw(K,triangle,ax,f'{name}: beta={betti(K)[:2]}')
plt.show()

Scale is held fixed here. The difference comes entirely from what higher-order relation is inferred from the same three pairwise edges. A graph retains the cycle, Čech retains it because the balls lack a joint intersection, and Rips fills it by clique completion.

## 5. Interpret: lecture demonstration

**Presenter close.** Ask what the example supports, what information was discarded and which stronger claim would be unjustified.

1. Rips fills every clique, while Čech requires a joint intersection. Pairwise overlap does not imply triple overlap.
2. For a good cover such as finite Euclidean balls and their non-empty intersections, the Nerve Theorem gives the nerve and union the same homotopy type.
3. A Rips complex contains a simplex for every clique, so the displayed proximity graph is only its 1-skeleton.
4. Clique filling treats mutual pairwise interaction as evidence of a coherent higher-order unit. This may or may not match the mechanism.
5. Euclidean distance may ignore periodic boundaries, anisotropic scales, observation functions, temporal order, or dynamically meaningful similarity.

**◇ Object check.** The observed object is a finite point cloud. The union of balls, its Čech nerve, the Rips clique complex, and their homology groups are four different constructed objects. A loop belongs to the selected construction, not automatically to the underlying dynamical system.

## Lecture close

Return to the final slide questions.

1. Name the observed or starting object.
2. Name every constructed object used in this walkthrough.
3. Identify the single modelling decision that drove the central comparison.
4. State one conclusion supported by the calculation and one conclusion it cannot establish.

**Take-forward example.** The purpose of three equal-radius balls followed by one sampled circle is to make Čech and Rips filtrations concrete. The example is deliberately small or synthetic so that the construction remains inspectable.